# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Research Question:** Which pre-decision search signals are associated with subsequent declines in page visibilty, and how can they be used to rank pages for human review?

**Decision it supports:** This work helps human reviwers prioritize which pages to review first and decide whether content or layout updates are warranted, using observed and directional evidence rather than intuition.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release:**  FlyRank warehouse release build v20260703 from Hugging Face.

**Table and grain:** I used the `fact_content_daily_performance` table, querying the February and March 2026 partitions. The working dataset contains one row per `client_hash_id` and `content_hash_id` pair after monthly aggregation.

**Feature window:** February 1–28, 2026. The ``prior_*` fields from this window were used as model features.

**Target window:** March 1–31, 2026. March impressions were used to calculate the observed `future_decline_label`.

**Excluded from features:** 
- March 2026 features were not added in training the model because they were what the model was tested on.
- `client_hash_id` and `content_hash_id` as predictive features, because they are pseudonymous identifiers. `client_hash_id` was retained only for grouped splitting and evaluation; `content_hash_id` was retained for identifying output rows. .
- `future_impressions` column: Excluded because it contains information from the target window and would cause data leakage if added to the model training data.
- `future_decline_label` column: Excluded because it is the target and as such can never be a feature

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

### Assumptions

- The working dataset contains one row per `client_hash_id` and `content_hash_id` pair, with February 2026 features and a March 2026 outcome.
- A row is eligible for labeling only when it has at least 100 February impressions. Rows without an observed label are excluded from model training and evaluation.
- Missing GA4 values may represent unavailable tracking rather than zero engagement.
- A missing February average position means that no measurable February search position was available; it does not mean rank zero.
- This is observational data. An observed decline does not prove that any feature caused the decline.

### Features

The model uses February-only fields that were available before the target window:

- `prior_impressions`
- `prior_clicks`
- `prior_ctr`, derived from February clicks divided by February impressions
- `prior_avg_position`
- `prior_sessions`
- `prior_engagement_rate`

Missing `prior_avg_position` values are filled with 999, while missing sessions and engagement values are filled with 0, following the modeling notebook's preprocessing. These values are used as preprocessing conventions, not as claims that the underlying measurements were truly zero.

### Label definition

`future_decline_label` is the observed March outcome. It equals 1 when a page has at least 100 February impressions and its March impressions are less than 80% of its February impressions; otherwise, eligible rows receive 0. `future_impressions` is used to construct and evaluate the label, but never as a feature.

### Baseline rule and reason codes

The transparent baseline ranks eligible client-content pairs by a rule-based estimate of future decline risk. Its score is:

```python
dataframe['baseline_score'] = (
    3 * dataframe['limited_prior_visibility']
    + dataframe['weak_position_signal']
    + dataframe['low_prior_engagement']
    + dataframe['low_click_through_rate']
)
```

The reason codes are:

- `limited_prior_visibility`: at least 100 but fewer than 1,000 prior impressions.
- `weak_position_signal`: prior average position is worse than 10.
- `low_prior_engagement`: prior engagement rate is below 30% when sessions are available.
- `low_click_through_rate`: prior clicks are low relative to prior impressions.
- `high_visibility_at_risk`: at least 1,000 prior impressions and a weak prior position; this is an interpretation of the `limited_prior_visibility` and position signals, not a separate term in the score.

A page can receive more than one reason code. The baseline is a transparent prioritization rule, not a causal model.

### Validation design

The logistic-regression model uses `GroupShuffleSplit` with a 70/30 train/test split, `client_hash_id` as the grouping variable, and `random_state=42`. Grouping keeps pages from the same client on only one side of the split, reducing client-specific memorization. Because this analysis uses one February-to-March month pair, it is not a full time-series validation across multiple target periods.

### Leakage checks

Only February `prior_*` fields are used as features. `future_impressions`, `future_decline_label`, and all other March metrics are excluded from the feature matrix because they contain target-window information. `client_hash_id` and `content_hash_id` are retained only for grouping, splitting, joining, and identifying output rows; they are not learned as predictive features. The label-derived fields `trend_direction` and `trend_pct` are also excluded.

## 4. Results (vs baseline)

The logistic-regression model and the transparent rule baseline were evaluated on the same 28,904-row, client-grouped held-out test set. Both methods reached 0.35 precision@20, compared with an 0.188 test-set decline base rate. The model therefore ties the baseline in this run; it does not demonstrate an improvement over the simpler rule.

The code below rebuilds the split, trains the model using February-only features, evaluates both rankings on the same test rows, and prints the honest comparison table.

In [13]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

# Find the repository root and load the cached February-March dataset.
repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents]:
    cache_file = candidate / 'work' / 'outputs' / 'february_march_features.parquet'
    if cache_file.exists():
        repo_root = candidate
        break

cache_path = repo_root / 'work' / 'outputs' / 'february_march_features.parquet'
dataframe = pd.read_parquet(cache_path)
dataframe = dataframe[dataframe['future_decline_label'].notna()].copy()

# Build only February features; March is used only as the observed outcome.
feature_frame = dataframe[[
    'prior_impressions', 'prior_clicks', 'prior_avg_position',
    'prior_sessions', 'prior_engagement_rate',
]].copy()
feature_frame['prior_avg_position'] = feature_frame['prior_avg_position'].fillna(999)
feature_frame['prior_sessions'] = feature_frame['prior_sessions'].fillna(0)
feature_frame['prior_engagement_rate'] = feature_frame['prior_engagement_rate'].fillna(0)
feature_frame['prior_ctr'] = (
    dataframe['prior_clicks']
    / dataframe['prior_impressions'].replace(0, np.nan)
).fillna(0)
model_features = [
    'prior_impressions', 'prior_clicks', 'prior_ctr',
    'prior_avg_position', 'prior_sessions', 'prior_engagement_rate',
]
feature_frame = feature_frame[model_features]
target = dataframe['future_decline_label'].to_numpy()
groups = dataframe['client_hash_id']

# Keep every page from a client on one side of the split.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_indices, test_indices = next(splitter.split(feature_frame, target, groups=groups))

scaler = StandardScaler()
train_features = scaler.fit_transform(feature_frame.iloc[train_indices])
test_features = scaler.transform(feature_frame.iloc[test_indices])
model = LogisticRegression(random_state=42, max_iter=1000, solver='lbfgs')
model.fit(train_features, target[train_indices])

test_dataframe = dataframe.iloc[test_indices].copy()
test_dataframe['model_probability'] = model.predict_proba(test_features)[:, 1]

# Recreate the transparent baseline on these same held-out rows.
test_dataframe['prior_ctr'] = feature_frame.iloc[test_indices]['prior_ctr'].to_numpy()
test_dataframe['weak_position_signal'] = (
    test_dataframe['prior_avg_position'] > 10
).fillna(False).astype(int)
test_dataframe['low_prior_engagement'] = (
    (test_dataframe['prior_sessions'] > 0)
    & (test_dataframe['prior_engagement_rate'] < 0.30)
).fillna(False).astype(int)
test_dataframe['low_click_through_rate'] = (
    test_dataframe['prior_ctr'] < 0.01
).fillna(False).astype(int)
test_dataframe['limited_prior_visibility'] = (
    test_dataframe['prior_impressions'] < 1000
).astype(int)
test_dataframe['baseline_score'] = (
    3 * test_dataframe['limited_prior_visibility']
    + test_dataframe['weak_position_signal']
    + test_dataframe['low_prior_engagement']
    + test_dataframe['low_click_through_rate']
)

ranked_model = test_dataframe.sort_values(
    ['model_probability', 'prior_impressions'], ascending=[False, True]
).reset_index(drop=True)
ranked_baseline = test_dataframe.sort_values(
    ['baseline_score', 'prior_impressions'], ascending=[False, True]
).reset_index(drop=True)
top_20_model = ranked_model.head(20)
top_20_baseline = ranked_baseline.head(20)
base_rate = test_dataframe['future_decline_label'].mean()
comparison_table = pd.DataFrame({
    'method': ['Rule baseline', 'Logistic Regression'],
    'precision_at_20': [
        top_20_baseline['future_decline_label'].mean(),
        top_20_model['future_decline_label'].mean(),
    ],
    'base_rate': [base_rate, base_rate],
    'test_rows': [len(test_dataframe), len(test_dataframe)],
})

print('Results on the same client-grouped held-out test set:')
print(comparison_table.to_string(index=False))
print(f"Model top-20 hits: {int(top_20_model['future_decline_label'].sum())}/20")
print(f"Baseline top-20 hits: {int(top_20_baseline['future_decline_label'].sum())}/20")
assert len(test_dataframe) == 28904
assert len(set(groups.iloc[train_indices]) & set(groups.iloc[test_indices])) == 0

Results on the same client-grouped held-out test set:
             method  precision_at_20  base_rate  test_rows
      Rule baseline             0.35    0.18769      28904
Logistic Regression             0.35    0.18769      28904
Model top-20 hits: 7/20
Baseline top-20 hits: 7/20


## 5. Limitations

This analysis supports directional prioritization, not causal or production claims. It uses one February-to-March month pair, so it cannot establish whether the ranking generalizes across seasons, algorithm changes, or other target windows. The client-grouped split protects against sharing a client's pages across train and test, but it is still a random holdout rather than a true out-of-time evaluation.

The model and baseline tie at precision@20 in this run, so the logistic regression does not justify replacing the simpler rule. Missing GA4 data may indicate unavailable tracking rather than zero engagement, and the label measures impression decline rather than business value, conversions, or content quality. High-risk rows must therefore receive human review before any content action.

The diagnostic cell reports the error rate, false positives, false negatives, uncertain predictions, and the share of eligible rows covered by the held-out queue. These measurements describe this evaluation sample; they are not guarantees for future data.

In [14]:
# Measure mistakes and uncertainty on the held-out rows only.
test_dataframe['model_prediction'] = (
    test_dataframe['model_probability'] >= 0.5
).astype(int)
test_dataframe['is_error'] = (
    test_dataframe['model_prediction']
    != test_dataframe['future_decline_label']
).astype(int)

error_summary = pd.DataFrame({
    'measure': [
        'eligible_rows', 'held_out_queue_rows', 'queue_coverage',
        'error_rate', 'false_negatives', 'false_positives',
        'low_confidence_rows',
    ],
    'value': [
        len(dataframe),
        len(test_dataframe),
        len(test_dataframe) / len(dataframe),
        test_dataframe['is_error'].mean(),
        int(((test_dataframe['model_prediction'] == 0)
             & (test_dataframe['future_decline_label'] == 1)).sum()),
        int(((test_dataframe['model_prediction'] == 1)
             & (test_dataframe['future_decline_label'] == 0)).sum()),
        int(test_dataframe['model_probability'].between(0.40, 0.60).sum()),
    ],
})
print(error_summary.to_string(index=False))

# A grouped split is valid only if no client appears in both partitions.
train_clients = set(groups.iloc[train_indices])
test_clients = set(groups.iloc[test_indices])
print(f"Client overlap between train and test: {len(train_clients & test_clients)}")
assert len(train_clients & test_clients) == 0

            measure        value
      eligible_rows 80322.000000
held_out_queue_rows 28904.000000
     queue_coverage     0.359852
         error_rate     0.187690
    false_negatives  5425.000000
    false_positives     0.000000
low_confidence_rows     2.000000
Client overlap between train and test: 0


## 6. Ranked recommendations

The recommended workflow is a human review queue, not an automatic publishing or client decision system. Reviewers should start with the model-ranked top 20, inspect the reason codes and underlying February metrics, and then choose an appropriate content or measurement follow-up.

Priority 1 is a high-visibility page with weak position or multiple warning signals because a fix may protect existing search demand. Priority 2 is a low-CTR or weak-position page that needs a search-result and intent review. Priority 3 is a limited-visibility page, where reviewers should first confirm that the query has sufficient demand before investing in a refresh.

The code counts the reason codes among the top 20 model-ranked rows and writes a compact action summary for the paper.

In [15]:
# Add human-readable reason codes to the model-ranked queue.
def build_reason_code(row):
    reasons = []
    if row['prior_impressions'] >= 1000 and row['prior_avg_position'] > 10:
        reasons.append('high_visibility_at_risk')
    if row['weak_position_signal']:
        reasons.append('weak_position_signal')
    if row['low_prior_engagement']:
        reasons.append('low_prior_engagement')
    if row['low_click_through_rate']:
        reasons.append('low_click_through_rate')
    if row['limited_prior_visibility']:
        reasons.append('limited_prior_visibility')
    return '; '.join(reasons) if reasons else 'no_flag_triggered'

ranked_model['reason_code'] = ranked_model.apply(build_reason_code, axis=1)
top_20_actions = ranked_model.head(20).copy()
reason_counts = (
    top_20_actions['reason_code']
    .str.split('; ')
    .explode()
    .value_counts()
)

action_map = {
    'high_visibility_at_risk': 'Review and refresh first; protect existing demand.',
    'weak_position_signal': 'Check search intent, content coverage, and internal links.',
    'low_prior_engagement': 'Check page experience, tracking availability, and engagement path.',
    'low_click_through_rate': 'Review title, description, and search-result intent match.',
    'limited_prior_visibility': 'Validate query demand before spending refresh effort.',
    'no_flag_triggered': 'Inspect raw metrics manually before taking action.',
}
action_summary = pd.DataFrame({
    'reason_code': reason_counts.index,
    'top_20_count': reason_counts.values,
})
action_summary['recommended_first_check'] = action_summary['reason_code'].map(action_map)
action_summary = action_summary.sort_values(
    'top_20_count', ascending=False
).reset_index(drop=True)

outputs_dir = repo_root / 'work' / 'outputs'
outputs_dir.mkdir(parents=True, exist_ok=True)
action_summary_path = outputs_dir / 'capstone_action_summary.csv'
action_summary.to_csv(action_summary_path, index=False)
print(action_summary.to_string(index=False))
print(f"\nSaved action summary to: {action_summary_path}")

             reason_code  top_20_count                                            recommended_first_check
    weak_position_signal            20         Check search intent, content coverage, and internal links.
  low_click_through_rate            20         Review title, description, and search-result intent match.
limited_prior_visibility            20              Validate query demand before spending refresh effort.
    low_prior_engagement             2 Check page experience, tracking availability, and engagement path.

Saved action summary to: C:\Users\DELL\Documents\flyrank-ml-internship-starter\work\outputs\capstone_action_summary.csv


## 7. Artifacts the paper embeds

The paper needs a small set of reproducible artifacts: one honest results table, one risk-distribution chart, and one reason-code chart. These artifacts show both the measured outcome and how a reviewer would interpret the ranked queue. The exports contain aggregate results or pseudonymous identifiers only; no client names, URLs, or raw queries are included.

In [16]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

figures_dir = repo_root / 'work' / 'figures'
figures_dir.mkdir(parents=True, exist_ok=True)

# Save the same comparison numbers shown in Section 4.
results_path = outputs_dir / 'capstone_results.csv'
comparison_table.to_csv(results_path, index=False)

# Chart 1: predicted decline probabilities in the honest queue.
probability_figure, probability_axis = plt.subplots(figsize=(6, 4))
probability_axis.hist(
    ranked_model['model_probability'],
    bins=20,
    color='#4C72B0',
    edgecolor='white',
)
probability_axis.axvline(0.5, color='gray', linestyle='--', linewidth=1)
probability_axis.set_xlabel('Predicted March decline probability')
probability_axis.set_ylabel('Number of pages')
probability_axis.set_title('Decline-risk distribution in the held-out queue')
probability_figure.tight_layout()
probability_path = figures_dir / 'capstone_probability_distribution.png'
probability_figure.savefig(probability_path, dpi=150)
plt.close(probability_figure)

# Chart 2: human-readable flags among the top 20 recommendations.
reason_figure, reason_axis = plt.subplots(figsize=(7, 4))
reason_counts.sort_values().plot(kind='barh', ax=reason_axis, color='#DD8452')
reason_axis.set_xlabel('Count among top 20 pages')
reason_axis.set_ylabel('Reason code')
reason_axis.set_title('Signals behind the top 20 recommendations')
reason_figure.tight_layout()
reason_path = figures_dir / 'capstone_top20_reason_codes.png'
reason_figure.savefig(reason_path, dpi=150)
plt.close(reason_figure)

artifact_manifest = pd.DataFrame({
    'artifact': [
        str(results_path.relative_to(repo_root)),
        str(action_summary_path.relative_to(repo_root)),
        str(probability_path.relative_to(repo_root)),
        str(reason_path.relative_to(repo_root)),
    ],
    'purpose': [
        'Model versus baseline results on the same held-out split',
        'Human review actions grouped by top-20 reason code',
        'Distribution of model probabilities in the honest queue',
        'Reason-code counts among the top-20 recommendations',
    ],
})
manifest_path = outputs_dir / 'capstone_artifact_manifest.csv'
artifact_manifest.to_csv(manifest_path, index=False)

print(artifact_manifest.to_string(index=False))
print(f"\nArtifacts written under {figures_dir.relative_to(repo_root)} and {outputs_dir.relative_to(repo_root)}")

                                          artifact                                                  purpose
                 work\outputs\capstone_results.csv Model versus baseline results on the same held-out split
          work\outputs\capstone_action_summary.csv       Human review actions grouped by top-20 reason code
work\figures\capstone_probability_distribution.png  Distribution of model probabilities in the honest queue
      work\figures\capstone_top20_reason_codes.png      Reason-code counts among the top-20 recommendations

Artifacts written under work\figures and work\outputs


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.